# Fine-Tuning LLaMA 3.2 Vision for Medical Image Understanding

## Aim
To adapt a pre-trained multimodal vision-language model
(LLaMA 3.2 Vision) to generate medically relevant descriptions
of radiology images using parameter-efficient fine-tuning (LoRA).

This work is for **educational and research purposes only**.

In [ ]:
# verify pyTorch sees the GPU

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")``

In [ ]:
!pip install pillow==11.3.0

In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer


In [ ]:
from huggingface_hub import login, logout
logout()
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Not logged in!


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model.gradient_checkpointing_enable()
model.config.use_cache = False

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
# 🔑 Memory-saving settings (correct place)
model.config.use_cache = False
model.gradient_checkpointing_enable()

In [ ]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.3 MB/s eta 0:00:00


In [ ]:
from medmnist import PneumoniaMNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = PneumoniaMNIST(
    split='train',
    transform=transform,
    download=True
)

100%|██████████| 4.17M/4.17M [00:01<00:00, 3.39MB/s]


In [ ]:
import numpy as np
from PIL import Image

LABEL_MAP = {
    0: "a normal chest X-ray with no visible abnormalities",
    1: "a chest X-ray showing signs consistent with pneumonia"
}

def format_example(idx):
    img, label = train_dataset[idx]

    # Extract scalar label safely
    label = int(label[0]) if hasattr(label, "__len__") else int(label)

    # Tensor → PIL Image (KEEP as PIL, NOT bytes)
    img_pil = Image.fromarray(
        (img.squeeze().numpy() * 255).astype(np.uint8)
    )

    return {
        # 🔑 Actual image goes here
        "images": [img_pil],

        # 🔑 Messages contain ONLY placeholders
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "You are a medical imaging assistant. "
                            "Describe this chest X-ray using clinical terminology. "
                            "Do not provide diagnosis or treatment."
                        )
                    },
                    {
                        "type": "image"
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": LABEL_MAP[label]
                    }
                ]
            }
        ]
    }


In [ ]:
from datasets import Dataset

train_data = [format_example(i) for i in range(150)]
hf_train_dataset = Dataset.from_list(train_data)

In [ ]:
hf_train_dataset[0].keys()

dict_keys(['images', 'messages'])

In [ ]:
# Apply LoRA (Parameter-efficient fine-tuning)

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                     # SAFE for free Colab
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,898,240 || all params: 10,676,119,075 || trainable%: 0.0552


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/medical_llama_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # ⬅️ increase this
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train_dataset,
    processing_class=processor,
    args=training_args
)

In [ ]:
trainer.train()


# ---------- AUTO SAVE ----------
SAVE_DIR = "/content/medical_llama_lora"

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("✅ LoRA adapters saved.")

# ---------- AUTO ZIP ----------
import shutil
zip_path = "/content/medical_llama_lora.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", SAVE_DIR)

print("✅ Model zipped.")

# ---------- AUTO DOWNLOAD ----------
from google.colab import files
files.download(zip_path)


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 12.12 MiB is free. Process 12920 has 14.73 GiB memory in use. Of the allocated memory 14.55 GiB is allocated by PyTorch, and 48.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## test the finetuned model

In [ ]:
sample = hf_train_dataset[0]["messages"]

inputs = processor.apply_chat_template(
    sample,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(processor.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')